In [21]:
# LIBRERIAS DE TRABAJO
#-----------------------------------
import pandas as pd
import requests
import os
import io
import unicodedata #para manejo de tildes

# Para etapa de ingesta
#-------------------------------------- 
from sqlalchemy import create_engine, text
from sqlalchemy.types import String
import getpass #para manejo de password seguro en conexion a Mysql
import sqlparse #diseñada para entender la anatomía de un archivo SQL, ignorar comentarios y separar las sentencias correctamente


In [49]:
# NAVEGA POR RELEASES de GITHUB PARA ENCONTRAR LOS ARCHIVOS EXISTENTES

user = "Marcelo-F-Martin"
repo = "PI_UA_pipeline_analisis_de_cobranza"

# URL de la API para la última release
api_url = f"https://api.github.com/repos/{user}/{repo}/releases/latest"

response = requests.get(api_url)
assets = response.json().get('assets', []) # [] es el segundo argumento del .get(). Significa: "Si por alguna razón la llave 'assets' no existe, no rompas el programa, devuélveme una lista vacía".
assets


[{'url': 'https://api.github.com/repos/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/releases/assets/381513573',
  'id': 381513573,
  'node_id': 'RA_kwDORolAKc4WvW9l',
  'name': 'Detalle.de.comisiones.Broker_1.xls',
  'label': None,
  'uploader': {'login': 'Marcelo-F-Martin',
   'id': 267367303,
   'node_id': 'U_kgDOD--zhw',
   'avatar_url': 'https://avatars.githubusercontent.com/u/267367303?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/Marcelo-F-Martin',
   'html_url': 'https://github.com/Marcelo-F-Martin',
   'followers_url': 'https://api.github.com/users/Marcelo-F-Martin/followers',
   'following_url': 'https://api.github.com/users/Marcelo-F-Martin/following{/other_user}',
   'gists_url': 'https://api.github.com/users/Marcelo-F-Martin/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/Marcelo-F-Martin/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/Marcelo-F-Martin/subscriptions',
   'organizations_url': 'htt

## Inicia proceso ETL
#### 1- Extracción:
#####   Obtención de datos crudos desde la fuente y almacenamiento en Dataframe de Pandas

In [50]:
df_definitivo_por_libro = []   
df_final = []

#====== INICIO 2º bucle: recorre hojas del archivo ===============================

for asset in assets:
    
    # Filtramos para que solo lea los archivos .xls
    if asset['name'].endswith('.xls'):
        print(f"Detectado archivo: {asset['name']}")
        
        # El campo 'browser_download_url' es el link directo
        
    hojas_consolidadas_por_libro = [] # se crea una lista para incorporar los df por cada hoja de cada libro, para luego agrupar todos.
    
    file_url = asset['browser_download_url']
    file_data = requests.get(file_url).content
    
    ## df_temp = pd.read_excel(io.BytesIO(file_data), engine='xlrd') -----se incorpora esta linea en la linea siguiente.

    hojas_brutas = pd.read_excel(io.BytesIO(file_data), engine='xlrd', sheet_name=None, header=None)

#====== INICIO 1º bucle: recorre hojas del archivo ===============================
        
    for nombre_hoja, df_bruto in hojas_brutas.items():
                   
        # Aquí 'df_bruto' es el DataFrame bruto de la hoja actual
        print(f"✅ Hoja '{nombre_hoja}' cargada correctamente. Filas brutas: {len(df_bruto)}")
        # Retornamos el diccionario para el siguiente paso del procesamiento

        print(f"  --- Buscando clave en Hoja '{nombre_hoja}' ---")
        
        encabezado_clave = "Periodo"
        indice_encabezado_fila = df_bruto[df_bruto.apply(lambda row: encabezado_clave in row.astype(str).values, axis=1)].index
        
        fila_inicio = indice_encabezado_fila[0]
        print(f"✅ Encabezado clave '{encabezado_clave}' encontrado en la Fila (índice 0): {fila_inicio}")

        fila_encabezado = df_bruto.iloc[fila_inicio]

        # 2b. 'col_inicio' será el índice de la columna donde se encuentra la clave
        # Esto es crucial para ignorar las columnas añadidas a la izquierda.
        
        col_inicio = fila_encabezado[fila_encabezado.astype(str) == encabezado_clave].index[0]
        print(f"  ✅ Columna de inicio de datos (índice 0): {col_inicio}")

        # Por eficiencia, si ya tienes el df_bruto, lo mejor es usarlo para el recorte:
        df_bruto.columns = df_bruto.iloc[fila_inicio] # Asignar la fila encontrada como nuevo encabezado
        df_limpio = df_bruto[fila_inicio + 1:].reset_index(drop=True) # Eliminar filas superiores

        df_final_hoja = df_limpio.copy()
        df_final_hoja['hoja_origen'] = nombre_hoja # se inserta nueva columna que identifica la Hoja de Origen

        if not df_final_hoja.empty:
            hojas_consolidadas_por_libro.append(df_final_hoja)
            print(f"  ✅ Hoja '{nombre_hoja}' añadida a la consolidación. Registros: {len(df_final_hoja)}")
        else:
            print(f"  Advertencia: Hoja '{nombre_hoja}' vacía después de la limpieza. No se añadió.")


       
    df_archivo_unificado = pd.concat(hojas_consolidadas_por_libro, ignore_index=True)
    df_definitivo_por_libro.append(df_archivo_unificado)
            
#====== FIN 1º bucle: recorre hojas del archivo ===============================
#====== FIN 2º bucle: recorre hojas del archivo ===============================

# Unificación final
df_final = pd.concat(df_definitivo_por_libro, ignore_index=True)


Detectado archivo: Detalle.de.comisiones.Broker_1.xls
✅ Hoja 'B0001' cargada correctamente. Filas brutas: 33008
  --- Buscando clave en Hoja 'B0001' ---
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Columna de inicio de datos (índice 0): 0
  ✅ Hoja 'B0001' añadida a la consolidación. Registros: 33000
Detectado archivo: Detalle.de.comisiones.Carlos.Diaz_1.xls
✅ Hoja 'AI002' cargada correctamente. Filas brutas: 787
  --- Buscando clave en Hoja 'AI002' ---
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Columna de inicio de datos (índice 0): 0
  ✅ Hoja 'AI002' añadida a la consolidación. Registros: 779
✅ Hoja 'AI003' cargada correctamente. Filas brutas: 754
  --- Buscando clave en Hoja 'AI003' ---
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Columna de inicio de datos (índice 0): 0
  ✅ Hoja 'AI003' añadida a la consolidación. Registros: 746
✅ Hoja 'AI013' cargada correctamente. Filas brutas: 640
  --- Buscando clave en Ho

In [51]:
df_final.head()

7,Periodo,Region,Asesor,Denominación Asesor,Broker,ID Broker,Fecha Operacion,Contrato,Rama,Especialidad,ID Cliente,Nombre Cliente,Comision,Valor Neto,Porcentaje Comision,Forma Pago,Numero Recibo,hoja_origen
0,202506,A,AE047,Sebastián Franco,B0001,Broker Health Co.,45819,GUPW9,Oftalmología,ESP22,CL01790L,Martin Acosta,"1730,9665","34619,33","0,05",efe,24807,B0001
1,202510,A,AE006,Patricia Vega,B0001,Broker Health Co.,45950,JN4TM,Oftalmología,ESP22,CL01759E,Patricia Aguirre,"1906,9085","38138,17","0,05",transf,78856,B0001
2,202511,A,AE006,Patricia Vega,B0001,Broker Health Co.,45984,WKB45,Oftalmología,ESP22,CL00210T,Grupo Aguirre SA,"1558,109","31162,18","0,05",transf,71496,B0001
3,202503,A,AE011,Héctor Farías,B0001,Broker Health Co.,45745,55T20,Pediatría,ESP08,CL00655F,Laboratorio Quiroga SA,"1673,154","16731,54","0,1",efe,81111,B0001
4,202508,A,AE026,Camila Paredes,B0001,Broker Health Co.,45878,VK0Q0,Oftalmología,ESP24,CL00439E,Martin Sosa,"1598,249","15982,49","0,1",transf,38615,B0001


#### 2- Exploración y Transformación

In [52]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Periodo              46000 non-null  object
 1   Region               46000 non-null  object
 2   Asesor               46000 non-null  object
 3   Denominación Asesor  46000 non-null  object
 4   Broker               46000 non-null  object
 5   ID Broker            46000 non-null  object
 6   Fecha Operacion      46000 non-null  object
 7   Contrato             46000 non-null  object
 8   Rama                 46000 non-null  object
 9   Especialidad         46000 non-null  object
 10  ID Cliente           46000 non-null  object
 11  Nombre Cliente       46000 non-null  object
 12  Comision             46000 non-null  object
 13  Valor Neto           46000 non-null  object
 14  Porcentaje Comision  46000 non-null  object
 15  Forma Pago           46000 non-null  object
 16  Nume

In [53]:
def limpiar_tilde(texto):
    # 1. Normaliza el texto a la forma NFD (Descomposición)
    # Esto separa la 'á' en 'a' + 'tilde combinable'
    texto_normalizado = unicodedata.normalize('NFD', texto)
    
    # 2. Filtra y mantiene solo los caracteres que no sean "marcas" de acento (Mn)
    texto_sin_acentos = "".join(
        c for c in texto_normalizado if unicodedata.category(c) != 'Mn'
    )
    
    return texto_sin_acentos

In [54]:
# 1_estandariza a minusculas
# 2_elimina espacios en blanco
# 3_reemplaza vacíos por '_'
# 4_aplica función definida para limpiar tildes
#________________________________________________

df_final.columns = df_final.columns.str.lower().str.strip().str.replace(' ','_')
df_final.columns = [limpiar_tilde(col) for col in df_final.columns]
df_final.columns

Index(['periodo', 'region', 'asesor', 'denominacion_asesor', 'broker',
       'id_broker', 'fecha_operacion', 'contrato', 'rama', 'especialidad',
       'id_cliente', 'nombre_cliente', 'comision', 'valor_neto',
       'porcentaje_comision', 'forma_pago', 'numero_recibo', 'hoja_origen'],
      dtype='object')

In [55]:
# seleccion de columnas relevantes a importar a la Base de Datos
lista_columnas_seleccionadas = ['periodo','asesor','fecha_operacion','contrato','comision', 'valor_neto', 'porcentaje_comision','forma_pago', 'numero_recibo', 'hoja_origen'  ]
df_final = df_final[lista_columnas_seleccionadas]

In [56]:
df_final.head()

,periodo,asesor,fecha_operacion,contrato,comision,valor_neto,porcentaje_comision,forma_pago,numero_recibo,hoja_origen
0,202506,AE047,45819,GUPW9,"1730,9665","34619,33","0,05",efe,24807,B0001
1,202510,AE006,45950,JN4TM,"1906,9085","38138,17","0,05",transf,78856,B0001
2,202511,AE006,45984,WKB45,"1558,109","31162,18","0,05",transf,71496,B0001
3,202503,AE011,45745,55T20,"1673,154","16731,54","0,1",efe,81111,B0001
4,202508,AE026,45878,VK0Q0,"1598,249","15982,49","0,1",transf,38615,B0001


In [68]:
# modifica tipo de datos. se comenta provisoriamente para evaluar necesidad de cambio de tipo de datos.

#df_final = df_final.astype({'fecha_operacion': , 'valor_neto': float, 'porcentaje_comision': float})
df_final['fecha_operacion'] = pd.to_datetime(df_final['fecha_operacion']).dt.date
df_final.info()

C:\Users\orglo\AppData\Local\Temp\ipykernel_15400\2234540419.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_final['fecha_operacion'] = pd.to_datetime(df_final['fecha_operacion']).dt.date


DateParseError: year 45819 is out of range: 45819, at position 0

In [11]:
#round(df_final.describe(),2)   se comenta esta linea porque los dtype son todos object y da un warning

In [58]:
# Verifica existencia de PK
df_final.nunique()

periodo                   24
asesor                    54
fecha_operacion          726
contrato                3010
comision               45936
valor_neto             45876
porcentaje_comision        6
forma_pago                 3
numero_recibo          21279
hoja_origen                9
dtype: int64

In [60]:
df_final.contrato.nunique()

3010

In [66]:
# Esta celda puede ser opcional para exportar el dataframe en CSV.

ruta_archivo = r"C:\Users\orglo\OneDrive\Desktop\marcelo\Proyectos\PI"
nombre_csv_salida = f"comisiones_consolidadas.csv"
ruta_salida_completa = os.path.join(ruta_archivo, nombre_csv_salida)
df_final.to_csv(ruta_salida_completa, index=False, encoding='utf-8')


In [14]:
# se limita a 10 registros para prueba de ingestion en mysql

#df_final = df_final.head(100)
#df_final

#### SQL - ejecuta scripts sql: 
##### 1- crea base de datos y tablas
##### 2- inserta datos en tabla temporal
##### 3- inserta datos en tabla fact
##### 4- crea vistas en BD capa_dos

In [65]:
# Verifique los datos de su entorno en MySQL
usuario = "root"
host = "localhost"
puerto = "3306"
bd_capa_uno = "cobranzas_capa_uno"
bd_capa_dos = "cobranzas_capa_dos"
tabla_temp = "temp_cobranzas"
df = df_final

url_sql_ddl = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/PI_UA_DDL_proyecto_03%2Btablas_dim%2Bcapa_dos.sql'
url_sql_insert_fact = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/PI_UA_DDL_proyecto_insert_en_fact.sql'
url_sql_vistas = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/PI_UA_capa_dos_vistas.01.sql'

try:
    # Descargar el contenido del archivo
    respuesta_1 = requests.get(url_sql_ddl)
    respuesta_2 = requests.get(url_sql_insert_fact)    
    respuesta_3 = requests.get(url_sql_vistas)    
    
    if respuesta_1.status_code == 200 and respuesta_2.status_code == 200 and respuesta_3.status_code == 200:
        script_sql_1 = respuesta_1.text
        print("✅ Script SQL 1 descargado correctamente desde GitHub.")

        script_sql_2 = respuesta_2.text
        print("✅ Script SQL 2 descargado correctamente desde GitHub.")

        script_sql_3 = respuesta_3.text
        print("✅ Script SQL 3 descargado correctamente desde GitHub.")

        
        # Conectar Ejecutar en MySQL
        
        password = getpass.getpass("Ingrese su contraseña de MySQL: ")    
        engine_create = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}") # crea una nueva bd
        engine_insert = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{bd_capa_uno}") # inserta datos en temp_cobranzas
        engine_vista = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{bd_capa_dos}") # crea vistas

        # 1.DDL crea base de datos y tablas        
        with engine_create.begin() as connection:
            
            # Dividimos por punto y coma para ejecutar sentencias individuales
            for statement in script_sql_1.split(';'):
                if statement.strip():
                    connection.execute(text(statement))

        # 2.Ingesta el dataframe en tabla temporal (truncate + insert)
        with engine_insert.begin() as connection:
            connection.execute(text(f"TRUNCATE TABLE {tabla_temp}"))
            print(f"🧹 Tabla '{tabla_temp}' limpiada (registros eliminados).")
            
            df.to_sql(
                name=tabla_temp,
                con=connection,
                if_exists='append', 
                index=False,
                chunksize=1000, 
                method='multi' # Para que sea masivo y no registro por registro
            )
           
        print(f"✅ Ingesta exitosa: {len(df)} registros insertados.")

        # 3.Pasa datos de tabla_temp a tabla_fact        
        with engine_insert.begin() as connection:
             
            # Dividimos por punto y coma para ejecutar sentencias individuales
            for statement in script_sql_2.split(';'):
                if statement.strip():
                    connection.execute(text(statement))
        
        print("🚀 Ejecución en MySQL finalizada exitosamente.")

        # 4.Crea vistas en BD cobranzas_capa_dos        
        with engine_vista.begin() as connection:
             
            # Dividimos por punto y coma para ejecutar sentencias individuales
            for statement in script_sql_3.split(';'):
                if statement.strip():
                    connection.execute(text(statement))
        
        print("🚀 Ejecución en MySQL finalizada exitosamente.")


        
    else:
        print(f"❌ Error al acceder al archivo Nº 1: Estado {respuesta_1.status_code}")
        print(f"❌ Error al acceder al archivo Nº 2: Estado {respuesta_2.status_code}")
        print(f"❌ Error al acceder al archivo Nº 3: Estado {respuesta_3.status_code}")

except Exception as e:
    print(f"❌ Error durante el proceso: {e}")

print(f"===================================================")
print(f"===================================================")

✅ Script SQL 1 descargado correctamente desde GitHub.
✅ Script SQL 2 descargado correctamente desde GitHub.
✅ Script SQL 3 descargado correctamente desde GitHub.


Ingrese su contraseña de MySQL:  ········


🧹 Tabla 'temp_cobranzas' limpiada (registros eliminados).
❌ Error durante el proceso: (pymysql.err.OperationalError) (1292, "Incorrect date value: '45819' for column 'fecha_operacion' at row 1")
[SQL: INSERT INTO temp_cobranzas (periodo, asesor, fecha_operacion, contrato, comision, valor_neto, porcentaje_comision, forma_pago, numero_recibo, hoja_origen) VALUES (%(periodo_m0)s, %(asesor_m0)s, %(fecha_operacion_m0)s, %(contrato_m0)s, %(comision_m0)s, %(valor_neto_m0)s, %(porcentaje_comision_m0)s, %(forma_pago_m0)s, %(numero_recibo_m0)s, %(hoja_origen_m0)s), (%(periodo_m1)s, %(asesor_m1)s, %(fecha_operacion_m1)s, %(contrato_m1)s, %(comision_m1)s, %(valor_neto_m1)s, %(porcentaje_comision_m1)s, %(forma_pago_m1)s, %(numero_recibo_m1)s, %(hoja_origen_m1)s), (%(periodo_m2)s, %(asesor_m2)s, %(fecha_operacion_m2)s, %(contrato_m2)s, %(comision_m2)s, %(valor_neto_m2)s, %(porcentaje_comision_m2)s, %(forma_pago_m2)s, %(numero_recibo_m2)s, %(hoja_origen_m2)s), (%(periodo_m3)s, %(asesor_m3)s, %(fecha_